In [6]:
from FID.fid_score import calculate_fid, get_precomputed_fid_scores_path, save_fid_stats_as_dict
import gzip
import numpy as np
import os
from PIL import Image
import pickle
import torch
import yaml

# Compute FID Scores

In [7]:
path = '/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/models/experiments/'


dataset = 'cifar10/'
ex_name = '20231004-213641_b99ee'


checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)
print(configs)

from utils.data_utils import get_data, get_gen

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

{'data': {'data_name': 'cifar10', 'num_clusters_data': 10}, 'globals': {'config_name': 'cifar10', 'eager_mode': False, 'results_dir': PosixPath('/cluster/home/jogoncalves/treevae/models/experiments'), 'save_model': True, 'seed': 42, 'wandb_logging': 'disabled'}, 'parser': {}, 'run_name': 'cifar10', 'training': {'activation': 'mse', 'aug_decisions_weight': 100, 'augment': True, 'augmentation_method': ['InfoNCE', 'instancewise_full'], 'batch_size': 256, 'compute_ll': False, 'decay_kl': 0.01, 'decay_lr': 0.1, 'decay_stepsize': 100, 'encoder': 'cnn2', 'grow': True, 'initial_depth': 1, 'inp_shape': 3072, 'intermediate_fulltrain': False, 'kl_start': 0.01, 'latent_dim': [64, 64, 64, 64, 64, 64], 'lr': 0.001, 'mlp_layers': [4096, 512, 512, 512, 512, 512], 'num_clusters_tree': 10, 'num_epochs': 150, 'num_epochs_finetuning': 0, 'num_epochs_intermediate_fulltrain': 0, 'num_epochs_smalltree': 150, 'prune': True, 'weight_decay': 1e-05}}
Files already downloaded and verified
Files already downloaded

In [8]:
trainset.dataset.data.shape

(50000, 32, 32, 3)

In [9]:
calculate_fid([testset.dataset.data, testset.dataset.data], batch_size=256, device='mps', dims=2048)

100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


-2.2168933355715126e-12

In [5]:
# precompute fid scores
get_precomputed_fid_scores_path(trainset.dataset.data, "cifar10", "train", batch_size=256, device='mps', dims=2048 )
get_precomputed_fid_scores_path(testset.dataset.data, "cifar10", "test", batch_size=256, device='mps', dims=2048 )

Saving FID statistics


100%|██████████| 196/196 [04:38<00:00,  1.42s/it]


Saving FID statistics


100%|██████████| 40/40 [00:52<00:00,  1.31s/it]


'FID/fid_stats_precomputed/fid_stats_cifar10_test.npz'

In [6]:
data_path1 = 'FID/fid_stats_precomputed/fid_stats_cifar10_train.npz'
data_path2 = 'FID/fid_stats_precomputed/fid_stats_cifar10_test.npz'

calculate_fid([data_path1, data_path2], batch_size=256, device='mps', dims=2048)

3.151766121907997

In [8]:
data_path1 = 'FID/fid_stats_precomputed/fid_stats_cifar10_train.npz'
calculate_fid([data_path1, data_path1], batch_size=256, device='mps', dims=2048)

-2.9558577807620168e-12

In [3]:
stats = save_fid_stats_as_dict(testset.dataset.data, batch_size=256, device='mps', dims=2048 )

Saving FID statistics


100%|██████████| 40/40 [00:50<00:00,  1.26s/it]


In [4]:
calculate_fid([stats, stats], batch_size=256, device='mps', dims=2048)

1.1368683772161603e-12

# Check if it is the same as the package:

In [5]:
import pytorch_fid 
from pytorch_fid import fid_score

Testset vs Testset

In [13]:
# save test set images in testset.dataset.data in data/cifar10_test folder

for i in range(len(testset.dataset.data)):
    img = Image.fromarray(testset.dataset.data[i])
    img.save('data/cifar10_test/'+str(i)+'.png')

In [10]:
# our implementation
calculate_fid([testset.dataset.data, testset.dataset.data], batch_size=256, device='mps', dims=2048)

100%|██████████| 40/40 [00:50<00:00,  1.26s/it]


-2.2168933355715126e-12

In [14]:
# pytorch-fid implementation
fid_score.calculate_fid_given_paths(['data/cifar10_test', 'data/cifar10_test'], 256, 'mps', 2048)

100%|██████████| 40/40 [00:50<00:00,  1.25s/it]


-1.8758328224066645e-12

--> both have very low differences as expected, because we compare two identical datasets with each other in both cases

# load images to compare two different image sets

In [15]:
# load images from data/generated_images folder
images = []
for filename in os.listdir('data/generated_images'):
    if filename.endswith(".png"):
        im = Image.open('data/generated_images/'+filename)
        images.append(np.array(im))
images = np.array(images)

# load images from data/cifar10_test folder
images2 = []
for filename in os.listdir('data/cifar10_test'):
    if filename.endswith(".png"):
        im = Image.open('data/cifar10_test/'+filename)
        images2.append(np.array(im))


In [16]:
# our implementation
calculate_fid([images, images2], batch_size=256, device='mps', dims=2048)

100%|██████████| 40/40 [00:51<00:00,  1.29s/it]
/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/FID/fid_score.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/pytorch-recipe_1690826037851/work/torch/csrc/utils/tensor_new.cpp:248.)
  images_tensor = torch.tensor(images_tensor)
100%|██████████| 40/40 [00:49<00:00,  1.24s/it]


255.12171607235035

In [17]:
# pytorch-fid implementation from original repo
fid_score.calculate_fid_given_paths(['data/generated_images', 'data/cifar10_test'], batch_size=256, device='mps', dims=2048)

100%|██████████| 40/40 [00:50<00:00,  1.26s/it]


255.12171607235177

identical results up to some rounding errors --> our implementation works